# Noise reduction with PESQ (Google Colab or Kaggle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-merino-hub/audiolab/blob/main/notebooks/denoising_pesq.ipynb)

Runs `scripts/evaluate_denoising.py` on the VoiceBank+DEMAND test set **with PESQ**, which needs a C
compiler to install and therefore runs here instead of on Windows. No GPU needed: everything is
frame-by-frame DSP on the CPU.

- **Colab**: *Runtime → Run all* (the default CPU runtime is fine).
- **Kaggle**: *Settings → Internet → On*. The last cell does not apply there: the CSV appears under
  *Output* (`/kaggle/working`).

## 1. Code and metrics

In [ ]:
!git clone -q https://github.com/victor-merino-hub/audiolab.git
%cd audiolab
!pip install -q -e ".[pesq]"
import pesq  # fails here if the package did not compile

## 2. Test set (~310 MB, CC BY 4.0, Valentini-Botinhao et al., University of Edinburgh)

First try: download it from [Edinburgh DataShare](https://datashare.ed.ac.uk/handle/10283/2791).
The repository may refuse requests from cloud servers; if it does, the cell prints the error and the
next one takes the test set from Google Drive instead: put `voicebank_testset.zip` (the folders
`clean_testset_wav/` and `noisy_testset_wav/` zipped together) in the root of *My Drive*.

In [ ]:
import shutil
import urllib.request
import zipfile
from pathlib import Path

VB = Path("data/voicebank")
VB.mkdir(parents=True, exist_ok=True)
FILES = {"clean_testset_wav.zip": "dec213d3-bf57-4777-9663-c24bdce92d5e",
         "noisy_testset_wav.zip": "13c1bfbf-14a6-41db-9b41-8f7310f01ad5"}

def download(name, uuid):
    url = f"https://datashare.ed.ac.uk/server/api/core/bitstreams/{uuid}/content"
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(request, timeout=120) as response, open(VB / name, "wb") as f:
            shutil.copyfileobj(response, f)
    except Exception as error:                      # e.g. HTTP Error 403: Forbidden
        print(f"{name}: {error}")
        return False
    print(f"{name}: {(VB / name).stat().st_size / 1e6:.0f} MB")
    return zipfile.is_zipfile(VB / name)

downloaded = all(download(name, uuid) for name, uuid in FILES.items())
print("Downloaded from DataShare" if downloaded else "DataShare failed: the next cell uses Google Drive")

In [ ]:
if not downloaded:
    from google.colab import drive
    drive.mount("/content/drive")
    source = Path("/content/drive/MyDrive/voicebank_testset.zip")
    assert source.exists(), "put voicebank_testset.zip in the root of your Google Drive and run this cell again"
    for name in FILES:
        (VB / name).unlink(missing_ok=True)       # leftovers of the failed download
    shutil.copy(source, VB / "voicebank_testset.zip")

for archive in VB.glob("*.zip"):
    with zipfile.ZipFile(archive) as z:
        z.extractall(VB)
    archive.unlink()

# Stop here if the test set is not complete
counts = {d: len(list((VB / d).glob("*.wav"))) for d in ("clean_testset_wav", "noisy_testset_wav")}
print(counts)
assert all(n == 824 for n in counts.values()), "test set incomplete"

## 3. Evaluate: PESQ for every method and frame length

SI-SDR and STOI are computed locally; here only PESQ, to keep the run short (Colab has 2 CPUs).

In [ ]:
import os
print(os.cpu_count(), "CPUs")
!python scripts/evaluate_denoising.py --metrics pesq --out data/results/denoising_pesq.csv

## 4. Download the results

In [ ]:
from google.colab import files
files.download("data/results/denoising_pesq.csv")